# Computational complexity – Visualization

This notebook analyzes the computational complexity of the plotting functions in `plots.py`, line by line and globally, using O(n) notation (S7), and measures their execution time.

Notation used in this notebook:
- **n**: number of sprouts (2,795).
- **F**: number of features (7).
- **K**: number of treatments (6).

In [1]:
# Make the project root importable, because this notebook is inside src/<module>/
import sys, os, inspect, timeit
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
# this lets the notebook find the src folder, which is two levels up
sys.path.append(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from src.preprocessing.cleaning import preprocess
from src.analysis.features import add_features, FEATURE_COLS
from src.visualization import plots

df = add_features(preprocess(os.path.join(ROOT, 'data', 'raw', 'cjs.csv')))

## `boxplot_tl_in`

In [2]:
# inspect.getsource shows the code of the function, so I can analyze it here
print(inspect.getsource(plots.boxplot_tl_in))

def boxplot_tl_in(df, path=None):
    """
    Boxplots of total length (TL) and number of internodes (IN) by treatment.

    Parameters
    ----------
    df : pandas.DataFrame
        Clean dataset.
    path : str, optional
        Where to save the figure.

    Returns
    -------
    matplotlib.figure.Figure
    """
    # Boxplots to compare TL and IN between treatments and look for outliers
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, col, title in zip(axes, ['TL', 'IN'],
                              ['Total length (TL) by treatment',
                               'Number of internodes (IN) by treatment']):
        sns.boxplot(x='TR', y=col, data=df, hue='TR', palette='Set2',
                    legend=False, dodge=False, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Treatment')
        ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    _save(fig, path)
    return fig



*n = number of sprouts.*

| Line | Code | Complexity |
|---|---|---|
| 1 | `plt.subplots(1, 2, ...)` | O(1) |
| 2 | `for ax, col, title in zip(...)`: 2 iterations | O(1) |
| 3 | `sns.boxplot(...)`: splits the n sprouts by treatment and computes the quartiles of each group, which requires sorting | O(n log n) |
| 4 | titles, labels and rotation | O(1) |
| 5 | `_save(fig, path)`: rendering, does not depend on n | O(1) |

**Global complexity:** **O(n log n)**, dominated by the computation of the quartiles.

## `boxplot_features`

In [3]:
print(inspect.getsource(plots.boxplot_features))

def boxplot_features(df, features, path=None):
    """
    Boxplots of each feature by treatment.

    Skewed features are drawn as log(1 + x), so the treated groups, which are
    concentrated near zero, can be compared with the controls.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset with the features.
    features : list of str
        Features to plot.
    path : str, optional
        Where to save the figure.

    Returns
    -------
    matplotlib.figure.Figure
    """
    n_rows = int(np.ceil(len(features) / 3))
    fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4.5 * n_rows))
    axes = axes.flatten()
    for ax, feature in zip(axes, features):
        data = df[['TR', feature]].dropna().copy()
        label = UNITS[feature]
        if feature in LOG_FEATURES:
            # log1p means log of 1 + x, I used it because most values are near zero and with a normal scale the boxes looked flat
            data[feature] = np.log1p(data[feature])
           

*n = number of sprouts; F = number of features (7).*

| Line | Code | Complexity |
|---|---|---|
| 1 | `plt.subplots(...)` | O(F) |
| 2 | `for ax, feature in zip(axes, features):` | F iterations |
| 3 | `df[['TR', feature]].dropna().copy()` | O(n) |
| 4 | `np.log1p(data[feature])` | O(n) |
| 5 | `sns.boxplot(...)`: quartiles per treatment | O(n log n) |

**Global complexity:** F × O(n log n) = **O(F · n log n)**, that is, **O(n log n)**.

## `boxplot_control_vs_treated`

In [4]:
print(inspect.getsource(plots.boxplot_control_vs_treated))

def boxplot_control_vs_treated(df, features, path=None):
    """
    Boxplots of each feature for control vs. treated sprouts.

    It replaces the scatter plot of Lab 2: it shows how much the two groups
    overlap for each feature.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset with the TR_code column and the features.
    features : list of str
        Features to plot.
    path : str, optional
        Where to save the figure.

    Returns
    -------
    matplotlib.figure.Figure
    """
    data = df.copy()
    data['group'] = np.where(data['TR_code'] >= 4, 'Control', 'Treated')
    n_rows = int(np.ceil(len(features) / 4))
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 4 * n_rows))
    axes = axes.flatten()
    for ax, feature in zip(axes, features):
        values = data[['group', feature]].dropna().copy()
        label = UNITS[feature]
        if feature in LOG_FEATURES:
            # log1p means log of 1 + x, I used it because most values are nea

*n = number of sprouts; m = number of columns (35); F = number of features (7).*

| Line | Code | Complexity |
|---|---|---|
| 1 | `data = df.copy()` | O(n · m) |
| 2 | `np.where(data['TR_code'] >= 4, ...)` | O(n) |
| 3 | `for ax, feature in zip(axes, features):` | F iterations |
| 4 | `dropna().copy()` and `np.log1p(...)` | O(n) |
| 5 | `sns.boxplot(...)`: quartiles of 2 groups | O(n log n) |

**Global complexity:** O(n · m) + F × O(n log n) = **O(n log n)**.

## `median_barplot`

In [5]:
print(inspect.getsource(plots.median_barplot))

def median_barplot(df, features, path=None):
    """
    Bar plot of the median of each feature by treatment.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset with the features.
    features : list of str
        Features to plot.
    path : str, optional
        Where to save the figure.

    Returns
    -------
    matplotlib.figure.Figure
    """
    # Table with the median of each feature for each treatment
    medians = df.groupby('TR', observed=True)[features].median()
    # Bar plot of the medians, one subplot for each feature
    n_rows = int(np.ceil(len(features) / 3))
    fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4.5 * n_rows))
    axes = axes.flatten()
    for ax, feature in zip(axes, features):
        bars = ax.bar(medians.index.astype(str), medians[feature],
                      color=sns.color_palette('Set2', len(medians)))
        ax.bar_label(bars, fmt='%.2f', fontsize=8)
        ax.set_title(TITLES[feature], fontsize=10)
        ax.set_

*n = number of sprouts; F = number of features (7); K = number of treatments (6).*

| Line | Code | Complexity |
|---|---|---|
| 1 | `df.groupby('TR')[features].median()`: splits the n sprouts by treatment and finds the median of each feature (sorting) | O(F · n log n) |
| 2 | `plt.subplots(...)` | O(F) |
| 3 | `for ax, feature in zip(axes, features):` | F iterations |
| 4 | `ax.bar(...)` and `ax.bar_label(...)`: K bars | O(K) |

**Global complexity:** O(F · n log n) + O(F · K) = **O(n log n)**. Drawing the bars does not depend on n.

## `radar_plot`

In [6]:
print(inspect.getsource(plots.radar_plot))

def radar_plot(df, features, path=None):
    """
    Radar plot of the median feature profile of each treatment (as in S4).

    Each feature is divided by its maximum median, so all axes go from 0 to 1.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset with the features.
    features : list of str
        Features to plot.
    path : str, optional
        Where to save the figure.

    Returns
    -------
    matplotlib.figure.Figure
    """
    # Radar plot of the median profile of each treatment, each feature is divided by its maximum so it goes from 0 to 1
    medians = df.groupby('TR', observed=True)[features].median()
    radar = medians / medians.max()
    N = len(features)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    plt.xticks(angles[:-1], features, size=9)
    ax.set_rlabe

*n = number of sprouts; F = number of features (7); K = number of treatments (6).*

| Line | Code | Complexity |
|---|---|---|
| 1 | `df.groupby('TR')[features].median()` | O(F · n log n) |
| 2 | `medians / medians.max()`: K × F table | O(K · F) |
| 3 | `angles = [...]` | O(F) |
| 4 | figure and axes setup | O(1) |
| 5 | `for i, treatment in enumerate(radar.index):` with `ax.plot` and `ax.fill` of F points | O(K · F) |

**Global complexity:** **O(n log n)**, from the medians; the drawing part only depends on K and F.

## Execution time of each plotting function

In [7]:
functions = {'boxplot_tl_in': lambda: plots.boxplot_tl_in(df),
             'boxplot_features': lambda: plots.boxplot_features(df, FEATURE_COLS),
             'boxplot_control_vs_treated': lambda: plots.boxplot_control_vs_treated(df, FEATURE_COLS),
             'median_barplot': lambda: plots.median_barplot(df, FEATURE_COLS),
             'radar_plot': lambda: plots.radar_plot(df, FEATURE_COLS)}

times = {}
for name, f in functions.items():
    times[name] = min(timeit.repeat(lambda: (f(), plt.close('all')), number=1, repeat=3))
pd.Series(times, name='time (s)').round(3)

boxplot_tl_in                 0.141
boxplot_features              0.467
boxplot_control_vs_treated    0.325
median_barplot                0.197
radar_plot                    0.023
Name: time (s), dtype: float64

**Discussion.** `boxplot_features` is the most expensive function (about 0.45 s), followed by `boxplot_control_vs_treated` and `median_barplot`. All plotting functions are **O(n log n)** because computing quartiles and medians requires sorting, but with n = 2,795 that computation takes only milliseconds. Most of the time is spent creating and rendering the figures, which depends on the number of subplots and drawn elements, not on n. That is why the functions with seven subplots are the slowest, and the radar plot, which draws only six polygons, is the fastest (about 0.02 s).